# SurgeShield — Model Training

**OSEMN stage: Model (M)**

This notebook narrates the modeling stage. The reproducible version is
`train.py`, and the shared transform lives in `preprocessing.py`. We import and
run `train.py` here so the notebook reports exactly what the script produces —
no separate, drift-prone copy of the logic.

**Three models, one per family:**
- Logistic Regression — linear baseline
- Random Forest — bagging ensemble
- XGBoost — boosting ensemble

**Protocol:** stratified 80/20 split → 5-fold stratified CV on the training set
for selection → score accuracy / precision / recall / F1 / ROC-AUC → pick the
best by cross-validated F1 → save artifacts to `models/`.

All metrics are the real, measured values. Given the Scrub/Explore finding (no
learnable signal), we expect every model to land near 0.50 ROC-AUC — and we
report that honestly rather than tuning toward a fabricated number.

## 1. Shared preprocessing

`preprocessing.py` defines one transform used by both training and the API:
standard-scale the 7 numeric features, pass through the 2 binary features, and
one-hot encode `Land Cover` and `Soil Type`. Inside cross-validation the
preprocessor is refit on each fold, so no test information leaks into training.

In [1]:
import json
import pandas as pd

import preprocessing as pp
import train

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print('Numeric    :', pp.NUMERIC_FEATURES)
print('Binary     :', pp.BINARY_FEATURES)
print('Categorical:', pp.CATEGORICAL_FEATURES)
print('Models     :', list(train.get_models().keys()))

Numeric    : ['Rainfall', 'Temperature', 'Humidity', 'River Discharge', 'Water Level', 'Elevation', 'Population Density']
Binary     : ['Infrastructure', 'Historical Floods']
Categorical: ['Land Cover', 'Soil Type']
Models     : ['Logistic Regression', 'Random Forest', 'XGBoost']


## 2. Train all three models

Running `train.main()` performs the split, 5-fold CV, held-out test evaluation,
selection by F1, and writes the four artifacts into `models/`. The console
output below is the live, measured result.

In [2]:
train.main()

SurgeShield — model training (Model)
[split] train=8,000  test=2,000  (stratified 80/20)
[prep] 19 features after encoding; transform paths agree


[eval] Logistic Regression    CV F1=0.5540  test F1=0.5450  test AUC=0.4984


[eval] Random Forest          CV F1=0.5224  test F1=0.5179  test AUC=0.4943


[eval] XGBoost                CV F1=0.5217  test F1=0.5039  test AUC=0.4913

[select] best by cross-validated F1: Logistic Regression

Held-out test metrics:
                     accuracy  precision  recall     f1  roc_auc
Logistic Regression    0.5025     0.5068  0.5895 0.5450   0.4984
Random Forest          0.5020     0.5071  0.5292 0.5179   0.4943
XGBoost                0.4950     0.5005  0.5074 0.5039   0.4913

[save] best_model.joblib  (Logistic Regression)
[save] scaler.joblib
[save] encoder.joblib
[save] all_models_comparison.json
Done. Metrics above are the real, measured values — not hard-coded.


## 3. Results — cross-validation (model selection)

We reload `all_models_comparison.json` (the saved source of truth that the
Flask API and analytics UI also read) and render it. Cross-validated means ± std
on the training folds — this is what selection is based on.

In [3]:
with open(train.MODELS_DIR / 'all_models_comparison.json', encoding='utf-8') as f:
    comp = json.load(f)

cv_rows = {
    name: {m: f"{d['cv'][m]['mean']:.4f} ± {d['cv'][m]['std']:.4f}" for m in train.SCORING}
    for name, d in comp['models'].items()
}
pd.DataFrame(cv_rows).T

,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.5139 ± 0.0059,0.5167 ± 0.0049,0.5971 ± 0.0131,0.5540 ± 0.0084,0.5186 ± 0.0094
Random Forest,0.5111 ± 0.0061,0.5162 ± 0.0058,0.5287 ± 0.0107,0.5224 ± 0.0081,0.5123 ± 0.0115
XGBoost,0.5129 ± 0.0112,0.5184 ± 0.0115,0.5252 ± 0.0110,0.5217 ± 0.0083,0.5156 ± 0.0140


## 4. Results — held-out test set (honest performance)

The 2,000-row test set was never seen during training or selection. These are
the numbers reported as the system's real performance.

In [4]:
test_df = pd.DataFrame({name: d['test'] for name, d in comp['models'].items()}).T
test_df.style.format('{:.4f}') if hasattr(test_df, 'style') else test_df.round(4)

,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.5025,0.5068,0.5895,0.5450,0.4984
Random Forest,0.5020,0.5071,0.5292,0.5179,0.4943
XGBoost,0.4950,0.5005,0.5074,0.5039,0.4913


In [5]:
print('Selected best model (by cross-validated F1):', comp['best_model'])
print('Saved artifacts in models/:')
for p in sorted(train.MODELS_DIR.glob('*')):
    print('  -', p.name)

Selected best model (by cross-validated F1): Logistic Regression
Saved artifacts in models/:
  - all_models_comparison.json
  - best_model.joblib
  - encoder.joblib
  - scaler.joblib


## 5. Conclusion

All three models perform at roughly chance level (ROC-AUC ≈ 0.50, F1 ≈ 0.50),
exactly as the signal-validation and EDA stages predicted. The differences
between models are within noise — there is no signal for a better algorithm to
exploit.

This is reported transparently: `best_model.joblib`, `scaler.joblib`,
`encoder.joblib`, and `all_models_comparison.json` capture the real outcome. The
next stage (`5_interpretation/`) turns these into the confusion matrix, ROC
curves, feature importances, and the `metrics.json` single source of truth that
the API and UI read — always real, never hard-coded.